In [ ]:
using Pkg
using Random
using Statistics
using Printf
using LinearAlgebra
using Logging

function find_project_root(start::AbstractString=pwd())
    active_project = Base.active_project()
    if !isnothing(active_project)
        active_root = dirname(active_project)
        if isfile(joinpath(active_root, "Project.toml")) && isfile(joinpath(active_root, "src", "System1D.jl"))
            return active_root
        end
    end

    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "System1D.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate project root from $start")
        dir = parent
    end
end

PROJECT_ROOT = find_project_root()
Pkg.activate(PROJECT_ROOT; io=devnull)

using Revise
using Plots

includet(joinpath(PROJECT_ROOT, "Experiments", "common", "notebook_helpers.jl"))

NOTEBOOK_REL_DIR = joinpath("Experiments", "systems", "periodic_ion_ring_1d", "gfmc", "notebooks")
PATHS = nb_paths(PROJECT_ROOT, NOTEBOOK_REL_DIR)
nb_include_formatting(PATHS.notebook_dir)

includet(joinpath(PROJECT_ROOT, "src", "System1D.jl"))
using .System1D
includet(joinpath(PATHS.notebook_dir, "periodic_ion_ring_helpers.jl"))

default(; dpi=170)
nothing


## Spinless Fermions on a Periodic Ion Ring

This notebook uses the generic GFMC Hamiltonian path for a closed-shell spinless-fermion ring in a periodic
soft-Coulomb ionic lattice. The lowest `N` exact one-body orbitals of the periodic Hamiltonian are assembled into
a real Slater determinant, and that determinant is used for:

- `TrialWF(log|Ψ_T|, ∇log|Ψ_T|, ∇²log|Ψ_T|, signΨ_T)`
- `ImportanceGuiding(trial, H)`
- `FixedNode()`
- `run_gfmc_with_vmc_init(...)`

Because the particles are noninteracting, the exact many-body benchmark is the sum of the lowest occupied one-body energies.


In [ ]:
N = 3
M = 3
a = 1.0
D = 0.5
ion_strength = 1.0
ion_softening = 0.35 * a
kmax = 6
quad_points = 1536

ring = build_periodic_ion_ring_model(
    M,
    a;
    D=D,
    ion_strength=ion_strength,
    ion_softening=ion_softening,
    kmax=kmax,
    quad_points=quad_points,
)

H = periodic_ion_hamiltonian(ring, N)
trial = fermion_determinant_trial_wavefunction(ring, N; node_tol=1.0e-10)
guiding = ImportanceGuiding(trial, H)
occupied = collect(1:N)
occupied_energies = ring.energies[occupied]
exact_energy = exact_manybody_energy(ring, N)

targetN = 256

vmc_dt = 3.0e-3
vmc_nsteps = 60
vmc_params = VMCParams(; dt=vmc_dt, nsteps=vmc_nsteps, targetN=targetN, ET0=exact_energy)

gfmc_dt = 1.0e-3
gfmc_nsteps = 250
gfmc_nequil = 50
feedback = 0.05
reconfiguration_interval = 2
branch_cap = 5.0
energy_window = 20
gfmc_params = GFMCParams(gfmc_dt, gfmc_nsteps, gfmc_nequil, targetN, exact_energy, feedback, reconfiguration_interval, branch_cap, energy_window)

rng_init = MersenneTwister(1234)
initial_positions = sample_uniform_ring_configurations(N, ring.L, targetN, rng_init; min_separation=0.02 * a)

MODEL_GRID_POINTS = 600
xgrid_model = Float64[i * (ring.L / MODEL_GRID_POINTS) for i in 0:(MODEL_GRID_POINTS - 1)]
onebody_potential_curve = Float64[onebody_potential(ring, x) for x in xgrid_model]

orbital_density_curves = Vector{Vector{Float64}}(undef, N)
dx_model = ring.L / MODEL_GRID_POINTS
for orb in 1:N
    phi_curve, _, _ = orbital_curve(ring, orb, xgrid_model)
    density_curve = phi_curve .^ 2
    density_curve ./= (sum(density_curve) * dx_model)
    orbital_density_curves[orb] = density_curve
end

exact_pooled_density = occupied_pooled_density(ring, occupied, xgrid_model; per_particle=true)
exact_pooled_density ./= (sum(exact_pooled_density) * dx_model)

SNAPSHOT_STEPS = nb_default_snapshot_steps(gfmc_nsteps)
DENSITY_GRID_POINTS = 400
DENSITY_BANDWIDTH = 0.08 * a
PAIR_SEP_BINS = 80
PERIOD_MARKERS = collect(0.0:a:ring.L)

RUN_LABEL = "exact-determinant fixed node"
RUN_COLOR = :navy
PLOT_TITLE = "Periodic ion ring GFMC (spinless fermions)"
MODEL_TRIAL_TITLE = "Periodic ion ring determinant-trial diagnostics"
DENSITY_TITLE = "Periodic ion ring GFMC: pooled fermion density evolution"
UNPOOLED_DENSITY_TITLE = "Periodic ion ring GFMC: final per-particle densities"
PAIR_TITLE = "Periodic ion ring GFMC: final pair-separation density"
PARTICLE_COLORS = [:navy, :darkorange, :forestgreen, :crimson, :purple, :goldenrod]

VMC_PROPOSAL = DriftGaussianProposal()
RECONFIGURATION = SystematicReconfiguration()

VMC_SHOW_PROGRESS = false
VMC_PROGRESS_EVERY = 0
VMC_DEBUG_MODE = false
VMC_DEBUG_EVERY = 10

SHOW_PROGRESS = false
PROGRESS_EVERY = 0
DEBUG_MODE = false
DEBUG_EVERY = 20

WRITE_RUN_CSV = false
CSV_FILENAME = "periodic_ion_ring_spinless_fermions_gfmc_vmc_init.csv"
SAVE_FIGURES = false
FIGURE_STEM = "periodic_ion_ring_spinless_fermions_gfmc_vmc_init"


In [ ]:
sim = run_gfmc_with_vmc_init(
    H,
    gfmc_params,
    initial_positions,
    trial,
    vmc_params;
    vmc_rng=MersenneTwister(41),
    gfmc_rng=MersenneTwister(52),
    proposal=VMC_PROPOSAL,
    guiding=guiding,
    nodepolicy=FixedNode(),
    reconfiguration=RECONFIGURATION,
    vmc_show_progress=VMC_SHOW_PROGRESS,
    vmc_progress_every=VMC_PROGRESS_EVERY,
    vmc_progress_label="VMC warm start",
    vmc_debug=VMC_DEBUG_MODE,
    vmc_debug_every=VMC_DEBUG_EVERY,
    snapshot_steps=SNAPSHOT_STEPS,
    show_progress=SHOW_PROGRESS,
    progress_every=PROGRESS_EVERY,
    progress_label=RUN_LABEL,
    debug=DEBUG_MODE,
    debug_every=DEBUG_EVERY,
)

start_idx = min(gfmc_params.nequil + 1, length(sim.energy_mean_history))
mean_energy, sem_energy = nb_mean_sem(sim.energy_mean_history[start_idx:end])

final_snapshot = nb_last_snapshot(sim)
final_pair_sep = Float64[]
for R in final_snapshot
    for i in 1:(length(R) - 1)
        for j in (i + 1):length(R)
            push!(final_pair_sep, distance_1d(ring.bc, R[i], R[j]))
        end
    end
end

println("GFMC step 0 corresponds to the VMC warm-start ensemble.")
println("occupied one-body energies = ", occupied_energies)
println(@sprintf("exact many-body energy = %.8f", exact_energy))
println(@sprintf("%s mean energy after nequil=%d: %.8f +/- %.3e", RUN_LABEL, gfmc_params.nequil, mean_energy, sem_energy))
println(@sprintf("post-equilibration energy error = %.3e", mean_energy - exact_energy))
println("final fixed walker count = ", sim.population_history[end])
println(@sprintf("final mean weight = %.6f", sim.mean_weight_history[end]))
println(@sprintf("final effective population = %.2f", sim.effective_population_history[end]))
println(@sprintf("minimum final pair separation = %.6f", minimum(final_pair_sep)))
println("count with r < 0.02a = ", count(r -> r < 0.02 * a, final_pair_sep))

if WRITE_RUN_CSV
    csv_path = joinpath(PATHS.tables_dir, CSV_FILENAME)
    nb_write_csv(csv_path, nb_gfmc_rows(RUN_LABEL, sim))
    println("Wrote run CSV to: ", abspath(csv_path))
end


In [ ]:
function pooled_ring_coordinates(snapshot)
    xs = Float64[]
    for R in snapshot
        append!(xs, Float64.(R))
    end
    return xs
end

function ring_particle_coordinates(snapshot, particle_idx::Integer)
    idx = Int(particle_idx)
    return Float64[R[idx] for R in snapshot]
end

function pair_separations(snapshot)
    rs = Float64[]
    for R in snapshot
        for i in 1:(length(R) - 1)
            for j in (i + 1):length(R)
                push!(rs, distance_1d(ring.bc, R[i], R[j]))
            end
        end
    end
    return rs
end

step0_snapshot = sim.walker_positions_history[1]
step0_xs = pooled_ring_coordinates(step0_snapshot)
step0_centers, step0_density = nb_periodic_kde_curve(
    step0_xs;
    xmin=0.0,
    xmax=ring.L,
    grid_points=DENSITY_GRID_POINTS,
    bandwidth=DENSITY_BANDWIDTH,
)

final_snapshot = nb_last_snapshot(sim)
final_xs = pooled_ring_coordinates(final_snapshot)
final_centers, final_density = nb_periodic_kde_curve(
    final_xs;
    xmin=0.0,
    xmax=ring.L,
    grid_points=DENSITY_GRID_POINTS,
    bandwidth=DENSITY_BANDWIDTH,
)

p_potential = plot(
    xgrid_model,
    onebody_potential_curve;
    xlabel="x",
    ylabel="V(x)",
    title="Periodic ion lattice potential",
    color=:black,
    linewidth=2.4,
    label="V_latt(x)",
    xlims=(0.0, ring.L),
)
for (k, xmark) in enumerate(PERIOD_MARKERS)
    vline!(p_potential, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "ion cell markers" : ""))
end

p_orbitals = plot(
    xlabel="x",
    ylabel="density",
    title="Occupied exact orbital densities",
    legend=:topright,
    xlims=(0.0, ring.L),
)
for orb in 1:N
    plot!(p_orbitals, xgrid_model, orbital_density_curves[orb]; linewidth=2.1, label="orbital $(orb)")
end
for (k, xmark) in enumerate(PERIOD_MARKERS)
    vline!(p_orbitals, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "ion cell markers" : ""))
end

p_density_compare = plot(
    xlabel="x",
    ylabel="density",
    title="Warm-start and GFMC density vs exact pooled density",
    legend=:topright,
    xlims=(0.0, ring.L),
)
plot!(p_density_compare, xgrid_model, exact_pooled_density; color=:black, linewidth=2.2, linestyle=:dash, label="exact pooled density")
plot!(p_density_compare, step0_centers, step0_density; color=RUN_COLOR, linewidth=2.4, label="step 0 density (after VMC)")
plot!(p_density_compare, final_centers, final_density; color=:crimson, linewidth=2.2, linestyle=:dot, label="final GFMC density")
for (k, xmark) in enumerate(PERIOD_MARKERS)
    vline!(p_density_compare, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "ion cell markers" : ""))
end

model_fig = plot(p_potential, p_orbitals, p_density_compare; layout=(3, 1), size=(1200, 1200), plot_title=MODEL_TRIAL_TITLE)
display(model_fig)
nb_save_figure(model_fig, PATHS.figures_dir, FIGURE_STEM, "model_trial"; enabled=SAVE_FIGURES)

history_fig = nb_plot_gfmc_history([sim]; labels=[RUN_LABEL], colors=[RUN_COLOR], title_prefix=PLOT_TITLE)
display(history_fig)
nb_save_figure(history_fig, PATHS.figures_dir, FIGURE_STEM, "history"; enabled=SAVE_FIGURES)

available_steps = SNAPSHOT_STEPS[1:min(length(SNAPSHOT_STEPS), length(sim.walker_positions_history))]
density_fig = plot(
    xlabel="x",
    ylabel="density",
    title=DENSITY_TITLE,
    legend=:topright,
    xlims=(0.0, ring.L),
)
for (snapshot, step_idx) in zip(sim.walker_positions_history, available_steps)
    xs = pooled_ring_coordinates(snapshot)
    centers, density = nb_periodic_kde_curve(
        xs;
        xmin=0.0,
        xmax=ring.L,
        grid_points=DENSITY_GRID_POINTS,
        bandwidth=DENSITY_BANDWIDTH,
    )
    step_label = step_idx == 0 ? "step 0 (after VMC warm start)" : "step $(step_idx)"
    plot!(density_fig, centers, density; label=step_label, color=RUN_COLOR, linewidth=2.2, alpha=0.82)
end
plot!(density_fig, xgrid_model, exact_pooled_density; color=:black, linewidth=2.0, linestyle=:dash, label="exact pooled density")
for (k, xmark) in enumerate(PERIOD_MARKERS)
    vline!(density_fig, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "ion cell markers" : ""))
end

display(density_fig)
nb_save_figure(density_fig, PATHS.figures_dir, FIGURE_STEM, "density"; enabled=SAVE_FIGURES)

particle_colors = [PARTICLE_COLORS[1 + mod(i - 1, length(PARTICLE_COLORS))] for i in 1:N]
unpooled_density_fig = plot(
    xlabel="x",
    ylabel="density",
    title=UNPOOLED_DENSITY_TITLE,
    legend=:topright,
    xlims=(0.0, ring.L),
)
for particle_idx in 1:N
    xs = ring_particle_coordinates(final_snapshot, particle_idx)
    centers, density = nb_periodic_kde_curve(
        xs;
        xmin=0.0,
        xmax=ring.L,
        grid_points=DENSITY_GRID_POINTS,
        bandwidth=DENSITY_BANDWIDTH,
    )
    plot!(unpooled_density_fig, centers, density; label="particle $(particle_idx)", color=particle_colors[particle_idx], linewidth=2.3)
end
for (k, xmark) in enumerate(PERIOD_MARKERS)
    vline!(unpooled_density_fig, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "ion cell markers" : ""))
end

display(unpooled_density_fig)
nb_save_figure(unpooled_density_fig, PATHS.figures_dir, FIGURE_STEM, "density_unpooled"; enabled=SAVE_FIGURES)

final_pair_sep = pair_separations(final_snapshot)
pair_centers, pair_density = nb_density_curve(
    final_pair_sep;
    nbins=PAIR_SEP_BINS,
    xmin=0.0,
    xmax=0.5 * ring.L,
    smoothing_window=9,
)
pair_fig = plot(
    pair_centers,
    pair_density;
    xlabel="r",
    ylabel="density",
    title=PAIR_TITLE,
    color=:darkorange,
    linewidth=2.4,
    label=false,
)
display(pair_fig)
nb_save_figure(pair_fig, PATHS.figures_dir, FIGURE_STEM, "pair_density"; enabled=SAVE_FIGURES)
